# Deep Learning Assignment 1
Submitters:
1. Amit Ner-Gaon 211649801
2. Chen Frydman 208009845

The purpose of this assignment is to build a simple neural network “from scratch” and to obtain a deep understanding of the forward/backward propagation process.

# Imports and Constants

In [1]:
import numpy as np

SEED = 42

# Section 1 - Forward Propagation

In [2]:
def initialize_parameters(layer_dims: list) -> dict:
    """
    Args:
        layer_dims (list): An array of the dimensions of each layer in the network (layer 0 is the size of the flattened input, layer L is the output softmax).
    Returns:
        parameters (dict): a dictionary containing the initialized W and b parameters of each layer {layer_number : {W,b}}.
    """
    rng = np.random.RandomState(SEED)  # for reproducibility

    parameters = {}  # {layer_number : {W,b}}

    L = (
        len(layer_dims) - 1
    )  # number of layers in the network (excluding the input layer)

    for l in range(
        1, L + 1
    ):  # starting from 1 because layer 0 is the input layer, which doesn't have parameters
        n_curr = layer_dims[l]
        n_prev = layer_dims[l - 1]

        current_weights = rng.normal(loc=0.0, scale=0.1, size=(n_curr, n_prev))

        parameters[l] = {
            "W": current_weights,
            "b": np.zeros((n_curr, 1)),  # bias is a vector of size n_curr
        }

    return parameters

In [3]:
def linear_forward(
    A: np.ndarray, W: np.ndarray, b: np.ndarray
) -> tuple[np.ndarray, dict]:
    """
    Args:
        A (np.ndarray): activations from previous layer (or input data): (size of previous layer, number of examples)
        W (np.ndarray): weights matrix, numpy array of shape (size of current layer, size of previous layer)
        b (np.ndarray): bias vector, numpy array of shape (size of the current layer, 1)
    Returns:
        Z (np.ndarray): the input of the activation function, also called pre-activation parameter
        linear_cache (dict): a python dictionary containing "A", "W" and "b" ; stored for computing the backward pass efficiently
    """
    Z = np.dot(W, A) + b  # Z = WA + b
    linear_cache: dict = {"A": A, "W": W, "b": b}
    return Z, linear_cache

$$Softmax(z)_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}$$

In [4]:
def softmax(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): The linear component of the activation function, of shape (number of classes, number of examples)
    Returns:
        A (np.ndarray): The output of the softmax activation function, of shape (number of classes, number of examples)
        activation_cache (dict): a python dictionary containing "Z", stored for computing the backward pass efficiently
    """
    shifted_Z = Z - np.max(
        Z, axis=0, keepdims=True
    )  # we substract maximum Z in order to avoid inf
    exp_Z = np.exp(shifted_Z)
    A = exp_Z / np.sum(
        exp_Z, axis=0, keepdims=True
    )  # axis=0 for sum down, keepdims for keeping the shape
    activation_cache: dict = {"Z": Z}
    return A, activation_cache

$$g(z)=max(0,z)$$
$$g'(z) = \begin{cases} 1 & \textit{if } z > 0 \\ 0 & \textit{if } z < 0 \end{cases}$$

In [5]:
def relu(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): the linear component of the activation function
    Returns:
        A (np.ndarray): the activations of the layer
        activiation_cache (dict): dict with Z
    """
    A = np.maximum(0, Z)
    activation_cache = {"Z": Z}
    return A, activation_cache

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [6]:
def sigmoid(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): the linear component of the activation function
    Returns:
        A (np.ndarray): the activations of the layer
        activiation_cache (dict): dict with Z
    """
    A = 1 / (1 + np.exp(-Z))
    activation_cache: dict = {"Z": Z}
    return A, activation_cache

In [7]:
def linear_activation_forward(
    A_prev: np.ndarray, W: np.ndarray, B: np.ndarray, activation: str
) -> tuple[np.ndarray, dict]:
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer
    Args:
        A_prev (np.ndarray): activations of the previous layer
        W (np.ndarray): the weights matrix of the current layer
        B (np.ndarray): the bias vector of the current layer
        activation (str): the activation function to be used (a string, either “softmax” or “relu”)
    Returns:
        A (np.ndarray):  the activations of the current layer
        cache (dict): a joint dictionary containing both linear_cache and activation_cache
    """
    if activation.lower() == "relu":
        Z, linear_cache = linear_forward(A_prev, W, B)
        A, activation_cache = relu(Z)
    elif activation.lower() == "sigmoid":
        Z, linear_cache = linear_forward(A_prev, W, B)
        A, activation_cache = sigmoid(Z)
    else:
        raise ValueError("Unsupported activation function. Use 'relu' or 'sigmoid'.")
    cache: dict = {"linear_cache": linear_cache, "activation_cache": activation_cache}
    return A, cache

In [ ]:
def l_model_forward(
    X: np.ndarray, parameters: dict, use_batchnorm: bool
) -> tuple[np.ndarray, list]:
    """
    Implement forward propagation for the [LINEAR->RELU]*(L-1)->LINEAR->SOFTMAX computation
    Args:
        X (np.ndarray): data, numpy array of shape (input size, number of examples)
        parameters (dict): the initialized W and b parameters of each layer
        use_batchnorm (bool): whether to use batch normalization or not
    Returns:
        AL (np.ndarray): last post-activation value
        caches (list): a list of all the cache objects generated by the linear_forward function
    """
    caches = []
    A = X
    L = len(parameters)  # number of layers in the neural network

    for l in range(1, L):  # loop from 1 to L-1
        A_prev = A
        W = parameters[l]["W"]
        b = parameters[l]["b"]
        A, cache = linear_activation_forward(A_prev, W, b, activation="relu")
        if use_batchnorm:
            A = apply_batchnorm(A)
        caches.append(cache)

    # Final layer with softmax activation
    W = parameters[L]["W"]
    b = parameters[L]["b"]
    AL, cache = linear_activation_forward(A, W, b, activation="softmax")
    caches.append(cache)

    return AL, caches

$$cost = -\frac{1}{m}\sum_{m=1}^{M}\sum_{c=1}^{C}y_i\log(\hat{y})$$

In [9]:
def compute_cost(AL: np.ndarray, Y: np.ndarray):
    """
    Args:
        AL (np.ndarray): probability vector corresponding to the label predictions, shape (number of classes, number of examples)
        Y (np.ndarray): the labels vector (i.e. the ground truth), shape (number of classes, number of examples)
    Returns:
        cost (float): the cross-entropy cost
    """
    m = Y.shape[1]  # number of examples
    cost: float = -np.sum(Y * np.log(AL + 1e-8)) / m  # adding epsilon to avoid log(0)
    return cost

In [10]:
def apply_batchnorm(A: np.ndarray) -> np.ndarray:
    """
    performs batchnorm on the received activation values of a given layer.
    Args:
        A (np.ndarray): the activations of a given layer, of shape (size of the layer, number of examples)
    Returns:
        NA (np.ndarray):  the normalized activation values, based on the formula learned in class
    """
    # axis = 1 for row-wise calculation, meaning for each feature across all the examples
    mu = np.mean(A, axis=1, keepdims=True)
    var = np.var(A, axis=1, keepdims=True)

    NA = (A - mu) / np.sqrt(var + 1e-8)

    return NA

# Section 2 - Backward Propagation

In [ ]:
def linear_backward(
    dZ: np.ndarray, cache: tuple[np.ndarray, np.ndarray, np.ndarray]
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Backward propagation process for a single layer.
    Args:
        dZ: the gradient of the cost with respect to the linear output of the **current** layer (layer l).
        cache: tuple of values (A_prev, W, b).
    Returns:
        (dA_prev, dW, db)
            dA_prev: Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev.
            dW: Gradient of the cost with respect to W (current layer l), same shape as W.
            db: Gradient of the cost with respect to b (current layer l), same shape as b.
    """

    A_prev, W, b = cache

    m = A_prev.shape[1]  # batch size. Used for averaging the gradients over the batch.

    # dz ((n{l}, m), a_prev (n{l-1}, m), w (n_l, n{l-1}), b (n_l, 1))
    dW = (1 / m) * np.dot(dZ, A_prev.T)
    db = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)

    return dA_prev, dW, db

In [ ]:
def relu_backward(dA: np.ndarray, activation_cache: dict) -> np.ndarray:
    """
    Backward propagation for a ReLU unit.
    Args:
        dA (np.ndarray): post activation gradient
        activation_cache (dict): contains Z
    Returns:
        dZ (np.ndarray): gradient of the cost with respect to Z
    """
    Z = activation_cache["Z"] if isinstance(activation_cache, dict) else activation_cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

In [ ]:
def softmax_backward(dA: np.ndarray, activation_cache: np.ndarray) -> np.ndarray:
    """
    Backward propagation for a softmax unit.
    Args:
        dA: post activation gradient
        activation_cache: contains Z
    Returns:
        dZ: gradient of the cost with respect to Z
    """
    # For softmax with cross-entropy, the derivative simplifies to AL - Y.
    Z = activation_cache["Z"] if isinstance(activation_cache, dict) else activation_cache
    dz = dA - Z

    return dz

In [ ]:
def linear_activation_backward(dA: np.ndarray, cache: dict, activation: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    The backward propagation for the LINEAR->ACTIVATION layer. The function
    first computes dZ and then applies the linear_backward function.
    Args:
        dA (np.ndarray): post activation gradient of the current layer
        cache (dict): contains both the linear cache and the activations cache.
    Returns:
        (dA_prev, dW, db):
            dA_prev (np.ndarray): Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev.
            dW (np.ndarray): Gradient of the cost with respect to W (current layer l), same shape as W.
            db (np.ndarray): Gradient of the cost with respect to b (current layer l), same shape as b.
    """
    linear_cache = cache["linear_cache"]
    activation_cache = cache["activation_cache"]

    if activation.lower() == "relu":
        dZ = relu_backward(dA, activation_cache)
    elif activation.lower() == "softmax":
        dZ = dA  # dZ already computed as AL - Y by the caller
    else:
        raise ValueError("Unsupported activation function. Use 'relu' or 'softmax'.")

    dA_prev, dW, db = linear_backward(
        dZ, (linear_cache["A"], linear_cache["W"], linear_cache["b"])
    )
    return dA_prev, dW, db

In [ ]:
def l_model_backward(AL: np.ndarray, Y: np.ndarray, caches: list) -> dict:
    """
    Backward propagation process for the entire network.
    The backpropagation for the softmax function is done only once (as only the output layers uses it).
    The backpropagation for the RELU is done iteratively over all the remaining layers of the network.
    Args:
        AL (np.ndarray): probability vector, output of the forward propagation (l_model_forward())
        Y (np.ndarray): true "label" vector (ground truth)
        caches (list): list of caches containing for each layer: a) the linear cache; b) the activation cache.

    Returns:
        grads: A dictionary with the gradients
                 grads["dA" + str(l)] = ...
                 grads["dW" + str(l)] = ...
                 grads["db" + str(l)] = ...
    """
    grads = {}
    L = len(caches)
    # Y = Y.reshape(AL.shape) # 

    # Output layer gradient (softmax + cross-entropy shortcut).
    dZL = softmax_backward(AL, Y)
    current_cache = caches[L - 1]
    dA_prev, dW, db = linear_activation_backward(dZL, current_cache, activation="softmax")
    grads["dA" + str(L - 1)] = dA_prev
    grads["dW" + str(L)] = dW
    grads["db" + str(L)] = db

    # Hidden layers (L-1 to 1) with ReLU.
    for l in reversed(range(L - 1)): # loop from L-2 to 0. index l much layer l + 1.
        current_cache = caches[l]
        dA_prev, dW, db = linear_activation_backward(
            grads["dA" + str(l + 1)], current_cache, activation="relu"
        )
        grads["dA" + str(l)] = dA_prev
        grads["dW" + str(l + 1)] = dW
        grads["db" + str(l + 1)] = db

    return grads

In [ ]:
def update_parameters(parameters: dict[np.ndarray], grads: dict[np.ndarray], learning_rate: float) -> dict[np.ndarray]:
    """
    Update parameters using gradient descent
    Args:
        parameters (dict[np.ndarray]): a python dictionary containing the DNN architecture’s parameters.
        grads (dict[np.ndarray]): a python dictionary containing the gradients (generated by L_model_backward)
        learning_rate (float): the learning rate (alpha).
    Returns:
        parameters (dict[np.ndarray]): python dictionary containing your updated parameters"""
    L = len(parameters)

    for l in range(1, L + 1):
        parameters[l]["W"] = parameters[l]["W"] - learning_rate * grads["dW" + str(l)]
        parameters[l]["b"] = parameters[l]["b"] - learning_rate * grads["db" + str(l)]

    return parameters

# Section 3 - Full Training Loop

In [ ]:
def l_layer_model(
    X: np.ndarray,
    Y: np.ndarray,
    layers_dims: list[int],
    learning_rate: float,
    num_iterations: int,
    batch_size: int,
    use_batchnorm: bool = False, # for section 5
) -> tuple[dict, list[float]]:
    """Implements a L-layer neural network. 
    All layers but the last have the ReLU activation function, and the final layer will apply the softmax activation function.
    The size of the output layer should be equal to the number of labels in the data. 
    
    Please select a batch size that enables your code to run well (i.e. no memory overflows while still running relatively fast).
    
    The function use earlier functions in the following order: initialize -> L_model_forward -> compute_cost -> L_model_backward -> update parameters

    Args:
        X (np.ndarray): the input data, a numpy array of shape (height*width , number_of_examples)   
            Input is in grayscale, so only one channel.
        Y (np.ndarray): One-hot labels of shape (n_classes, number_of_examples).
        layers_dims (list[int]): a list containing the dimensions of each layer, including the input.
        batch_size (int):  the number of examples in a single training batch.
        use_batchnorm (bool): whether to use batch normalization.

    Returns:
        Tuple[parameters, costs]:
            parameters (dict[int, dict[str, np.ndarray]]): (layer_number : {W,b}) the parameters learnt by the system during the training (the same parameters that were updated in the update_parameters function).
            costs (list[float]): the values of the cost function (calculated by the compute_cost function). One value is to be saved after each 100 training iterations (e.g. 3000 iterations -> 30 values).
    """
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def predict(X: np.ndarray, Y: np.ndarray, parameters: dict) -> float:
    """
    The function receives an input data and the true labels and calculates the accuracy of
    the trained neural network on the data.

    Args:
        X (np.ndarray): the input data, a numpy array of shape (height*width, number_of_examples)
        Y (np.ndarray): One-hot vector, the “real” labels of the data, a vector of shape (num_of_classes, number of examples)
        parameters (dict): (layer_number : {W,b}) python dictionary containing the DNN architectures parameters.


    Returns:
        accuracy (float): the percentage of the samples for which the correct label receives the hughest confidence
        score. 

    """
    raise NotImplementedError("This function is not implemented yet.")

# Section 4 - Experiments

## Section 4.a - Data Preparation

The MNIST dataset is publicly available at http://yann.lecun.com/exdb/mnist/ and consists of the following four parts:

- Training set images: train-images-idx3-ubyte.gz (9.9 MB, 47 MB unzipped, 60,000 examples)
- Training set labels: train-labels-idx1-ubyte.gz (29 KB, 60 KB unzipped, 60,000 labels)
- Test set images: t10k-images-idx3-ubyte.gz (1.6 MB, 7.8 MB, 10,000 examples)
- Test set labels: t10k-labels-idx1-ubyte.gz (5 KB, 10 KB unzipped, 10,000 labels)

In [ ]:
from sklearn.datasets import fetch_openml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

Download the MNIST dataset

In [ ]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True) #  returns (data, target). Data: Rows = samples, Columns = features.
print(f"X type: {type(X)}")
print(f"y type: {type(y)}")
X = X.values
y = y.astype(int).values

print("MNIST dataset loaded.")
print(f"X type: {type(X)}")
print(f"X shape: {X.shape}")
print(f"y type: {type(y)}")
print(f"y shape: {y.shape}")

Normalize to [-1, 1] range:


In [ ]:
X = ((X / 255.) - .5) * 2

Split into training, validation, and test set:

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.2, shuffle=True, stratify=y_temp)

Flatten input

In [ ]:
X_train = X_train.reshape(X_train.shape[0], 784)
X_valid = X_valid.reshape(X_valid.shape[0], 784)
X_test = X_test.reshape(X_test.shape[0], 784)

One Hot Encoding

In [ ]:
lb = LabelBinarizer()

lb.fit(y)  # Fit the label binarizer on all the labels to learn the classes.

y_train_oh = lb.transform(y_train)
y_valid_oh = lb.transform(y_valid)
y_test_oh  = lb.transform(y_test)

## Section 4.b - Training Configuration (Documentation)

Network parameters

In [ ]:
layer_dims = [784, 20, 7, 5, 10] # including the input layer
use_batchnorm = False
learning_rate = 0.009

Train the network

## Section 4.c - Required Report Outputs

## Section 4.d - Submission Format

# Section 5 - Conclusions